# Spatial Phenotype Interaction Analysis

This notebook summarizes which rule-based phenotypes are spatial neighbors within each ROI image. The analysis combines two outputs from earlier workflow stages:

```text
results/13_cells_with_phenotypes.csv
data/neighbors/*.csv
```

The phenotyped cell table provides one phenotype label per segmented cell. The neighbor tables provide cell-cell spatial edges from Steinbock. Joining these two data sources makes it possible to count phenotype-neighbor pairs, such as plasma-cell-like cells near T-cell-like or myeloid-like cells.

Important interpretation boundary:

- Neighbor edges represent spatial proximity in the segmented image.
- Neighbor edges do not prove direct cell-cell communication.
- Phenotype labels are rule-based approximations.
- This analysis is exploratory and should not be interpreted as the template paper's final prognostic interaction model.


## Step 0: Configure Workflow Paths

This setup cell defines paths for the phenotyped cell table, neighbor tables, summary outputs, and heatmap figure.

Main inputs:

```text
results/13_cells_with_phenotypes.csv
data/neighbors/*.csv
```

Main outputs:

```text
results/17_spatial_phenotype_interactions_by_image.csv
results/18_spatial_phenotype_interactions_by_category.csv
results/19_spatial_phenotype_interaction_summary.csv
figures/06_spatial_phenotype_interaction_heatmap.png
```


In [ ]:
from pathlib import Path
import csv
import subprocess

current = Path.cwd().resolve()
if current.name == 'notebooks':
    WORKFLOW_DIR = current.parent
else:
    WORKFLOW_DIR = current

DATA_DIR = WORKFLOW_DIR / 'data'
RESULTS_DIR = WORKFLOW_DIR / 'results'
FIGURES_DIR = WORKFLOW_DIR / 'figures'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'

PHENOTYPED_CELLS = RESULTS_DIR / '13_cells_with_phenotypes.csv'
NEIGHBORS_DIR = DATA_DIR / 'neighbors'

print('Workflow directory:', WORKFLOW_DIR)
print('Phenotyped cell table exists:', PHENOTYPED_CELLS.exists())
print('Neighbor directory exists:', NEIGHBORS_DIR.exists())
print('Neighbor files:', len(list(NEIGHBORS_DIR.glob('*.csv'))) if NEIGHBORS_DIR.exists() else 0)
print('Spatial script exists:', (SCRIPTS_DIR / '11_spatial_phenotype_interactions.py').exists())


## Step 1: Understand Spatial Phenotype Interactions

Steinbock neighbor tables contain rows with two cell object labels:

```text
Object, Neighbor
```

Each row means that two segmented objects are spatially close according to the neighbor rule used earlier:

```bash
steinbock measure neighbors --type expansion --dmax 4
```

The neighbor method expands cell boundaries by up to 4 pixels. If expanded cell regions touch, the cells are recorded as neighbors.

For phenotype interaction analysis, each cell label is mapped to its rule-based phenotype. A cell-cell edge then becomes a phenotype-phenotype edge.

Example:

```text
Object 10 phenotype: Plasma_cell_like
Object 25 phenotype: CD8_T_cell_like
Neighbor edge: 10 -- 25
Phenotype edge: Plasma_cell_like -- CD8_T_cell_like
```

Directed and undirected edge handling:

- Steinbock neighbor files can contain both `1 -> 2` and `2 -> 1`.
- For interaction counting, those two rows represent the same undirected spatial relationship.
- The script collapses duplicated reciprocal edges within each image before counting phenotype pairs.

Phenotype-pair ordering:

- Phenotype pairs are stored in canonical alphabetical order.
- This means `Plasma_cell_like -- T_cell_like` and `T_cell_like -- Plasma_cell_like` are counted as the same interaction type.


## Step 2: Run Spatial Phenotype Interaction Summary

The script performs the following operations:

```text
1. Read the phenotyped single-cell table.
2. Build a lookup from Image + Object to phenotype and category.
3. Read each neighbor table from data/neighbors/.
4. Collapse reciprocal directed edges into unique undirected edges.
5. Convert cell-cell edges into phenotype-phenotype edges.
6. Summarize phenotype-pair counts by image.
7. Summarize phenotype-pair counts by category.
8. Save a global phenotype interaction heatmap.
```

The image-level table is useful for ROI-specific inspection. The category-level table is useful for broader exploratory summaries across `MGUS`, `UB`, and `B`.


In [ ]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '11_spatial_phenotype_interactions.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--cells', 'results/13_cells_with_phenotypes.csv',
    '--neighbors-dir', 'data/neighbors',
    '--output-image', 'results/17_spatial_phenotype_interactions_by_image.csv',
    '--output-category', 'results/18_spatial_phenotype_interactions_by_category.csv',
    '--output-summary', 'results/19_spatial_phenotype_interaction_summary.csv',
    '--heatmap', 'figures/06_spatial_phenotype_interaction_heatmap.png',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)


## Step 3: Review Run Summary

The summary table records technical counts from the spatial interaction step.

Important metrics:

- `neighbor_files`: number of ROI neighbor tables processed.
- `raw_directed_edges`: number of rows read from the Steinbock neighbor files.
- `unique_undirected_edges`: number of non-duplicated spatial edges after reciprocal edge collapse.
- `edges_skipped_missing_cell`: edges skipped because one or both object labels were not found in the phenotyped cell table.
- `image_level_interaction_rows`: number of image-level phenotype-pair rows written.

A high skipped-edge count would indicate mismatch between neighbor files and the phenotyped cell table. A skipped-edge count near zero indicates consistent object identifiers across files.


In [ ]:
summary_path = RESULTS_DIR / '19_spatial_phenotype_interaction_summary.csv'
with summary_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        print(row)


## Step 4: Review Image-Level Phenotype Interactions

The image-level table reports phenotype-pair counts separately for each ROI image.

Columns:

```text
Image: ROI image name without .tiff
phenotype_a: first phenotype in the canonical pair
phenotype_b: second phenotype in the canonical pair
edge_count: number of unique neighbor edges for this phenotype pair
fraction_of_group_edges: fraction of all unique neighbor edges in that image
```

This table is useful for identifying ROI-specific spatial patterns and for checking whether a phenotype interaction appears consistently or only in selected images.


In [ ]:
image_interactions = RESULTS_DIR / '17_spatial_phenotype_interactions_by_image.csv'
with image_interactions.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

for row in rows[:30]:
    print(row)


## Step 5: Review Category-Level Phenotype Interactions

The category-level table aggregates phenotype-pair counts across images belonging to the same category.

Columns:

```text
category: sample category inferred from image names
phenotype_a: first phenotype in the canonical pair
phenotype_b: second phenotype in the canonical pair
edge_count: number of unique neighbor edges for this phenotype pair
fraction_of_group_edges: fraction of all unique neighbor edges in that category
```

This table supports exploratory comparison across `MGUS`, `UB`, and `B`. Because the representative subset is small and phenotype labels are rule-based approximations, this output should be treated as descriptive rather than inferential.


In [ ]:
category_interactions = RESULTS_DIR / '18_spatial_phenotype_interactions_by_category.csv'
with category_interactions.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

for row in rows[:40]:
    print(row)


## Step 6: Review Heatmap Output

The heatmap summarizes phenotype-phenotype neighbor edge counts aggregated across categories. It provides a compact visual overview of which phenotype pairs are frequently adjacent.

Interpretation guidance:

- Strong diagonal values indicate frequent same-phenotype adjacency.
- Off-diagonal values indicate mixed-phenotype spatial neighborhoods.
- Frequent edges involving `Unknown` indicate that unclassified cells remain spatially important and may require improved phenotyping rules.
- High interaction counts can reflect high abundance, spatial clustering, or both.

Further normalization may be needed before biological interpretation. Raw edge counts are useful for QC and exploration, but they are not enrichment statistics.


In [ ]:
heatmap = FIGURES_DIR / '06_spatial_phenotype_interaction_heatmap.png'
print('Heatmap exists:', heatmap.exists())
print('Heatmap size_bytes:', heatmap.stat().st_size if heatmap.exists() else 0)
